In [ ]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time

### Constants

In [ ]:
# function name
str_function_name = 'christian-concat-tuning'

### 1. Create container

### Create ```Dockerfile```

In [ ]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

### Write ```requirements.txt```

In [ ]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0
pandas==1.2.4
boto3==1.24.59

### Write ```lambda_function.py```

In [ ]:
%%writefile lambda_function.py

import pandas as pd
import numpy as np
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20240509-christian-internship'
    str_model = '09_aws_batch'
    str_prefix = f'{str_model}/output'

    # # get df_hyperparameters
    # str_filename = 'df_hyperparameters.csv'
    # str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/05_step_function/{str_filename}'
    # df = pd.read_csv(str_uri)
    # # convert to dict
    # dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # # get eval metric
    # str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']

    # hard code eval metric
    str_eval_metric = 'AUC'
    print(f'Eval metric: {str_eval_metric}')
    
    # get index files in s3
    print('Getting files in s3...')
    cls_client = boto3.resource('s3')
    cls_bucket = cls_client.Bucket(str_project)
    list_str_filenames = []
    for file in cls_bucket.objects.filter(Prefix=str_prefix):
        # get key
        str_key = file.key
        # make sure it is a .csv
        if '.csv' in str_key:
            # get filename
            str_filename = str_key.split('/')[-1]
            list_str_filenames.append(str_filename)
    print(f'There are {len(list_str_filenames)} files to import')
    
    # iterate and import
    print('Importing files...')
    list_df = []
    for str_filename in list_str_filenames:
        str_uri = f's3://{str_project}/{str_prefix}/{str_filename}'
        df = pd.read_csv(str_uri)
        list_df.append(df)
    
    # create df
    print('Creating data frame...')
    df = pd.concat(list_df)
    del list_df
    
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC', 'PRAUC', 'F1']:
        bool_ascending = False
    else:
        bool_ascending = True
        
    # sort
    df.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)
    
    # write to s3
    print('Writing to s3...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/08_aws_lambda/output/{str_filename}'
    df.to_csv(str_uri, index=False)

### Build image and push to ECR

In [ ]:
%%sh

# name the image
image=christian-concat-tuning

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

### 2. Create lambda function from image

In [ ]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [ ]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

In [ ]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

In [ ]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/christian-concat-tuning:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

### Clean-up

In [ ]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)